# Week 12 Supervised Learning - Classification

Earlier this week we learned the basics of a simple classifier, K-NN. We learned about train-test splits and how to evaluate models. Today we will extend this to more robust evaluation techniques, and how to "tune" hyperparameters. We will briefly cover other potential classifiers that could replace the K-NN algorithm. 

First, let's load in and prepare our data. We are again using the `ny_census` data.

In [ ]:
import pandas as pd
import geopandas as gpd 
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, RobustScaler 
from sklearn.compose import make_column_transformer # This is a function for transforming all columns
from sklearn.compose import make_column_selector # This allows us to specify what types of columns to scale
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [ ]:
# Let's load in our new york census data
df = pd.read_csv('ny_census.csv')
df['GEOID'] = df['GEOID'].astype(str) # We already have the GEOID but it's saved as a float

In [ ]:
df.iloc[0] # We have a lot of information about each census tract

In [ ]:
# Let's grab our outlines for plotting

year = 2020
state_fips = "36" #NY

url = f"https://www2.census.gov/geo/tiger/TIGER{year}/TRACT/tl_{year}_{state_fips}_tract.zip"
tracts = gpd.read_file(url)
tracts = tracts.to_crs(epsg=4326)
tracts.head()

# Merge with our census data
ny_df_tracts = pd.merge(left=df, right=tracts, on='GEOID', how='left')
ny_df_tracts.head()

In [ ]:
# Let's calculate density per square mile
## ALAND is in square meters right not so we need to convert it
ny_df_tracts['ALAND_sqmi'] = ny_df_tracts['ALAND']/2.59e+6
ny_df_tracts['density_sqmi'] = ny_df_tracts['TotalPopE']/ny_df_tracts['ALAND_sqmi']

In [ ]:
ny_df_tracts.density_sqmi.describe()

In [ ]:
# Let's calculate a few columns of interest

def urb_rur_midsize(x):
    if x > 25000:
        return "urban"
    elif x <= 25000 and x > 7000:
        return "midsize"
    else:
        return "rural"
# urban/rural designation
ny_df_tracts['urban_rural'] = ny_df_tracts['density_sqmi'].apply(lambda x: urb_rur_midsize(x))
# percent white in each tract
ny_df_tracts['perc_white'] = ny_df_tracts['WhiteE']/ny_df_tracts['TotalPopE']
# percent below the poverty line in each tract
ny_df_tracts['perc_poverty'] =  ny_df_tracts['PovertyE']/ny_df_tracts['TotalPopE']

In [ ]:
# convert to geodataframe and plot
ny_df_tracts = gpd.GeoDataFrame(ny_df_tracts, 
                                geometry='geometry',
                                crs='EPSG:4326')
ny_df_tracts.plot(column='urban_rural', categorical=True, legend=True, cmap='viridis')

### Recall our KNN code

In [ ]:
# Keep non NA rows, and separate into training/testing sets
df_tmp = ny_df_tracts[['urban_rural', 'perc_white', 'MedIncomeE']].dropna()
df_train, df_test = train_test_split(df_tmp, train_size=0.75)

# Recall our preprocessor
preprocessor = make_column_transformer((RobustScaler(), #run the robust scaler
                                        make_column_selector(dtype_include="number")), # only on columns that are numbers
                                       remainder="passthrough", # pass through every other column
                                       verbose_feature_names_out=False) # stops long names

In [ ]:
# And recall our KNN model
knn = KNeighborsClassifier(n_neighbors=5)

In [ ]:
# Now we can build them into one pipeline so data always gets preprocessed the same way 
knn_pipeline = make_pipeline(preprocessor, knn)


In [ ]:
# Let's fit our model
knn_pipeline.fit(X=df_train[["perc_white", "MedIncomeE"]], y=df_train["urban_rural"])


In [ ]:
# Now we can predict on our test data using the same pipeline
predicted = knn_pipeline.predict(df_test[["perc_white", "MedIncomeE"]])
df_test['predicted'] = predicted # Add new column
df_test[["urban_rural", "predicted"]].head(10) # Show side by side

In [ ]:
# How well do we do? This returns "accuracy"
knn_pipeline.score(df_test[['perc_white', 'MedIncomeE']], df_test['urban_rural'])

## 1. Cross Validation

Every time we split our data into a training and a testing set, we make a decision about which data should be considered to build our model (training). As we saw before, each time we run this process, we can end up with a slightly different training set and this can impact our results. To combat this, we could run the train-test split process multiple times, look at the accuracy results, and take the average to get a sense of how our model performs regardless of the noise in the training set. 

In practice, we don’t have to rely on random splits, but rather we can use a more structured splitting procedure so that each observation in the data set is used in a validation set only a single time. The name for this strategy is **cross-validation**. In cross-validation, we split our overall training data into evenly sized chunks. Then, iteratively use chunk as the validation set and combine the remaining chunks as the training set. In the figure below, different chunks of the data set are used, resulting in 5 different choices for the validation set; we call this 5-fold cross-validation.
</figure>
<img src="https://python.datasciencebook.ca/_images/cv.png" alt="drawing" width="500" style="display: block; margin: 0 auto"/>
</figure>


In [ ]:
from sklearn.model_selection import cross_validate

# Let's define our pipeline from scratch
knn_pipeline = make_pipeline(preprocessor, knn)

# We define the model AND fit it with data at the same time
cv_results = cross_validate(estimator=knn_pipeline,  # We pass the pipeline in 
                            cv=5, # We choose how many splits we want
                            X=df_tmp[["perc_white", "MedIncomeE"]], # X data for fitting
                            y=df_tmp["urban_rural"]) # y data for fitting
cv_results

In [ ]:
cv_results = pd.DataFrame(cv_results)['test_score']

# We can compute two main statistics: mean and standard error of the mean (sem)
cv_5_metrics = cv_results.agg(["mean", "sem"])
cv_5_metrics

`sem` is the standard error of the mean. This quantifies how precisely the sample mean estimates the true population mean, which gives more of an understanding of uncertainty. The `sem` is calculated as $\frac{\sigma}{\sqrt{n}}$ (the standard deviation over the square root of the number of folds)

In [ ]:
## YOUR TURN
## Run cross validation with 10 folds. Compute the mean and standard error. What is different from the 5-fold CV?



## 2. Choosing K

How do you choose the best number of neighbors?

Since cross-validation helps us evaluate the accuracy of our classifier, we can use cross-validation to calculate an accuracy for each value of in a reasonable range, and then pick the value of that gives us the best accuracy. The `scikit-learn` package collection provides built-in functionality, named `GridSearchCV`, to automatically handle the details for us. Before we use `GridSearchCV`, we need to create a new pipeline with a `KNeighborsClassifier` that has the number of neighbors left unspecified.

In [ ]:
knn = KNeighborsClassifier()
knn_tune_pipeline = make_pipeline(preprocessor, knn)

# We can see all of the hyperparameters we can tune
knn_tune_pipeline.get_params()

In [ ]:
# We specificy the values we want to search through in a dictionary
parameter_grid = {"kneighborsclassifier__n_neighbors": range(1, 200, 5)}


In [ ]:
from sklearn.model_selection import GridSearchCV

# Now we define the model so we can run through all of K
knn_tune_grid = GridSearchCV(estimator=knn_tune_pipeline, # this is our pipeline where K is not specified
                             param_grid=parameter_grid, # these are the values of K we want to try
                             cv=10) # With 10 folds in cross validation

In [ ]:
# Now we pass it the data
knn_tune_grid.fit(df_tmp[["perc_white", "MedIncomeE"]],df_tmp["urban_rural"])

# We need to get the results through .cv_results_
accuracies_grid = pd.DataFrame(knn_tune_grid.cv_results_)
accuracies_grid.head()

In [ ]:
# We can look at the uncertainty again
accuracies_grid["sem_test_score"] = accuracies_grid["std_test_score"] / 10**(1/2)

accuracies_grid[["param_kneighborsclassifier__n_neighbors", "mean_test_score", "sem_test_score"]]

In [ ]:
# Plot the K values against the mean test scores
plt.plot(accuracies_grid['param_kneighborsclassifier__n_neighbors'], accuracies_grid['mean_test_score'], '.-')
plt.ylabel('Mean Score')
plt.xlabel('Number of Neighbors (k)')


Generally, when selecting (and other parameters for other predictive models), we are looking for a value where:
* we get roughly optimal accuracy, so that our model will likely be accurate
* changing the value to a nearby one (e.g., adding or subtracting a small number) doesn’t decrease accuracy too much, so that our choice is reliable in the presence of uncertainty
* the cost of training the model is not prohibitive (e.g., in our situation, if is too large, predicting becomes expensive!)

## 3. Over- and underfitting

When we are training a classifier, we have to be careful about over- or under-fitting the training data. 

**Underfitting** is when our classifer under preforms and does not model the data well. Usually this is the case if you don't have enough training data or your model is too loose. In KNN, this might happen if you have too many neighbors who "get a say" in the value of your unknown data point. Too many neighbors can create an averaging effect where you lose the intricacies of your input data's relationship to the target values. 

**Overfitting** is when our classifier performs very well on our training set, but performs poorly on our test set because the model is too finetuned to the exact training data. In KNN, this happens when you have too few neighbors, such that each new point is only considering a few of its closest other points. The boundary between the classes becomes jagged and cannot account for what might come in the testing set or in the real world. 

</figure>
<img src="https://miro.medium.com/v2/resize:fit:1400/0*jB3VzCwWSwGXUX82.png" alt="drawing" width="700" style="display: block; margin: 0 auto"/>
</figure>


In [ ]:
# Let's look at our grid search if we bump the max neighbors up to 1000

large_param_grid = {"kneighborsclassifier__n_neighbors": range(1, 1000, 10),}

# 10-fold cross validation for each value of K
large_knn_tune_grid = GridSearchCV(estimator=knn_tune_pipeline,
                                   param_grid=large_param_grid,
                                   cv=10)

#Fit our data
large_knn_tune_grid.fit(df_tmp[["perc_white", "MedIncomeE"]],df_tmp["urban_rural"])

# Save the results in a dataframe
large_accuracies_grid = pd.DataFrame(large_knn_tune_grid.cv_results_)


In [ ]:
# Plot the K values against the mean test scores
plt.plot(large_accuracies_grid['param_kneighborsclassifier__n_neighbors'], large_accuracies_grid['mean_test_score'], '.-')
plt.ylabel('Mean Score')
plt.xlabel('Number of Neighbors (k)')

## 4. Briefly: Other Classifiers

Other classifiers: https://scikit-learn.org/stable/auto_examples/classification/plot_classifier_comparison.html


</figure>
<img src="https://scikit-learn.org/stable/_images/sphx_glr_plot_classifier_comparison_001.png" alt="drawing" width="900" style="display: block; margin: 0 auto"/>
</figure>



In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Create train-test
df_tmp = ny_df_tracts[['urban_rural', 'perc_white', 'MedIncomeE']].dropna()
df_train, df_test = train_test_split(df_tmp, train_size=0.75)

# Define the model and pipeline
dt = DecisionTreeClassifier()
dt_pipeline = make_pipeline(preprocessor, dt)

# Fit the data
dt_pipeline.fit(df_train[['perc_white', 'MedIncomeE']], df_train['urban_rural'])

# Predict on the test
df_test['predictions'] = dt_pipeline.predict(df_test[['perc_white', 'MedIncomeE']])

# Get accuracy results
print(classification_report(df_test["urban_rural"], df_test["predictions"]))
pd.crosstab(df_test["urban_rural"], df_test["predictions"])


In [ ]:
## YOUR TURN
## Pick another classifer from the link above and go through the modeling process with our data
